In [3]:
import os
import numpy as np
import pandas as pd
import supervision as sv
from supervision.metrics import MeanAveragePrecision

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [4]:
# Configuration files

# Combined Dataset to m0 Val
model_config = {
    'train': 'Combined Dataset',
    'test': 'm0 Val',
    'backbone': 'resnet50',
    'head': 'rhino',
    'config_file': 'configs-mine/rhino-resnet/rhino_phc_haus-4scale_r50_2xb2-36e_combined.py',
    'checkpoint_file': 'work_dirs/rhino_phc_haus-4scale_r50_2xb2-36e_combined',
    'val_dir': 'data/m0/val',
    'inf_dir': 'results/train_combined_test_m0',
    'img_height': 640,
    'epoch': 75,
}

In [3]:
def get_image_names_from_directory(directory):
    """Extracts image names (without extension) from a directory."""
    return {file_name.replace(".txt", "") for file_name in os.listdir(directory) if file_name.endswith(".txt")}

def load_detections(annotations_path, img_names, is_gt=True):
    """Loads detections only for images that exist in both GT and Predictions."""
    sv_data = []

    for image_id in sorted(img_names):
        file_path = os.path.join(annotations_path, f"{image_id}.txt")
        if not os.path.exists(file_path):  # Ensure file exists before processing
            continue

        xyxy_list = []
        class_ids = []
        scores = []

        with open(file_path, "r") as file:
            lines = file.readlines()

        for line in lines:
            data = list(map(float, line.split()))
            class_id = int(data[0])
            polygon = np.array(data[1:9]).reshape(4, 2)  # Convert to (4,2) shape
            score = data[9] if not is_gt else 1.0  # Default confidence for GT is 1.0

            # Convert quadrilateral to bounding box (min x, min y, max x, max y)
            x_min, y_min = np.min(polygon, axis=0)
            x_max, y_max = np.max(polygon, axis=0)
            bbox = [x_min, y_min, x_max, y_max]

            # Append to lists
            xyxy_list.append(bbox)
            class_ids.append(class_id)
            scores.append(score)

        # Convert lists into a Supervision Detections object
        detections = sv.Detections(
            xyxy=np.array(xyxy_list),
            class_id=np.array(class_ids),
            confidence=np.array(scores),
            metadata={"image_id": image_id}
        )

        sv_data.append(detections)

    return sv_data

def get_class_counts(detections_list, num_classes=3):
    """Counts occurrences of each class in ground truth detections."""
    class_counts = np.zeros(num_classes)
    for detections in detections_list:
        unique, counts = np.unique(detections.class_id, return_counts=True)
        for cls, count in zip(unique, counts):
            class_counts[cls] += count
    return class_counts

In [ ]:
index = pd.MultiIndex.from_tuples([], names=["Base State", "Target State",  "Epochs"])
result_df = pd.DataFrame(columns=["CFCBK", "FCBK", "Zigzag", "Weighted mAP@50", "mAP@50:95", "mAP@50", "mAP@75", "CA mAP@50:95", "CA mAP@50", "CA mAP@75"], index=index)

In [ ]:
for epoch in range(1, model_config['epoch'] + 1):
    # Load image names from directories
    GT_PATH = os.path.join(model_config['val_dir'], "labels")
    if not os.path.exists(GT_PATH):
        print(f"GT path {GT_PATH} does not exist.")
        break
    PREDICTIONS_PATH = os.path.join(model_config['inf_dir'], f"epoch_{epoch}", "annfiles")
    if not os.path.exists(PREDICTIONS_PATH):
        print(f"Predictions path {PREDICTIONS_PATH} does not exist.")
        continue
    gt_img_names = get_image_names_from_directory(GT_PATH)
    pred_img_names = get_image_names_from_directory(PREDICTIONS_PATH)
    img_names = gt_img_names.intersection(pred_img_names)
    base_state = model_config['train']
    target_state = model_config['test']

    # Load GT and Predictions
    gt_data = load_detections(GT_PATH, img_names, is_gt=True)
    pred_data = load_detections(PREDICTIONS_PATH, img_names, is_gt=False)

    # Print mAP results
    print(f"\n{model_config['train']} to {model_config['test']} (Epoch {epoch}):")
    ## mAP calculation (non-class agnostic)
    mAP_metric = MeanAveragePrecision(class_agnostic=False)
    mAP_result = mAP_metric.update(pred_data, gt_data).compute()
    matched_classes = mAP_result.matched_classes.tolist()
    # print(f"    Matched classes: {matched_classes}")
    # Extract overall mAP values
    mAP_50_95 = mAP_result.map50_95  # mAP 50:95
    mAP_50 = mAP_result.map50  # mAP 50
    mAP_75 = mAP_result.map75  # mAP 75
    print(f"    mAP 50:95: {mAP_50_95}, mAP 50: {mAP_50}, mAP 75: {mAP_75}")

    # Extract class-wise mAP
    class_wise_mAP = mAP_result.ap_per_class[:, 0].tolist()  # mAP 50:95 per class
    num_classes = 3
    final_class_wise_mAP = [0] * num_classes
    for cls, mAP in zip(matched_classes, class_wise_mAP):
        final_class_wise_mAP[cls] = mAP
    print(f"    class_wise_mAP: {final_class_wise_mAP}\n")
    # Calculate weighted mAP
    class_counts = get_class_counts(gt_data, num_classes=num_classes)
    print(f"    class_counts: {class_counts}")
    weighted_mAP_50 = np.sum(np.array(final_class_wise_mAP) * class_counts) / np.sum(class_counts)
    print(f"    Weighted mAP 50: {weighted_mAP_50}\n")

    # Compute class-agnostic mAP
    mAP_metric_agnostic = MeanAveragePrecision(class_agnostic=True)
    mAP_result_agnostic = mAP_metric_agnostic.update(pred_data, gt_data).compute()
    # Extract class-agnostic mAP values
    mAP_50_95_agnostic = mAP_result_agnostic.map50_95  # mAP 50:95
    mAP_50_agnostic = mAP_result_agnostic.map50  # mAP 50
    mAP_75_agnostic = mAP_result_agnostic.map75  # mAP 75
    print(f"    CA mAP 50:95: {mAP_50_95_agnostic}, CA mAP 50: {mAP_50_agnostic}, CA mAP 75: {mAP_75_agnostic}")

    # Update results dataframe
    result_df.loc[(base_state, target_state, epoch), :] = [f"{x:.6f}" for x in final_class_wise_mAP + [weighted_mAP_50, mAP_50_95, mAP_50, mAP_75, mAP_50_95_agnostic, mAP_50_agnostic, mAP_75_agnostic]]


In [ ]:
display(result_df)

In [ ]:
# Save the dataframe as CSV
result_df.to_csv(f"{model_config['train']}_{model_config['test']}_{model_config['head']}_{model_config['backbone']}_epoch_results.csv")

In [10]:
saved_df = pd.read_csv(f"{model_config['train']}_{model_config['test']}_{model_config['head']}_{model_config['backbone']}_epoch_results.csv", index_col=[0, 1, 2])
display(saved_df)

CFCBK      FCBK    Zigzag  \
Base State       Target State  Epochs                                 
Combined Dataset m0 Validation 1       0.126720  0.234075  0.397319   
                               2       0.385243  0.277017  0.584394   
                               3       0.380393  0.275270  0.640089   
                               4       0.519128  0.395661  0.552435   
                               5       0.363296  0.408457  0.658437   
...                                         ...       ...       ...   
                               71      0.406500  0.541084  0.768730   
                               72      0.410820  0.531358  0.774023   
                               73      0.418921  0.536678  0.782358   
                               74      0.406500  0.533601  0.769183   
                               75      0.410820  0.535119  0.777749   

                                       Weighted mAP@50  mAP@50:95    mAP@50  \
Base State       Target State  Epochs                                         
Combined Dataset m0 Validation 1              0.341634   0.086351  0.252704   
                               2              0.495879   0.167190  0.415551   
                               3              0.533804   0.183284  0.431917   
                               4              0.510882   0.215572  0.489075   
                               5              0.579441   0.210856  0.476730   
...                                                ...        ...       ...   
                               71             0.691873   0.332100  0.572105   
                               72             0.693302   0.323987  0.572067   
                               73             0.700859   0.318808  0.579319   
                               74             0.690288   0.326067  0.569762   
                               75             0.696840   0.322663  0.574563   

                                         mAP@75  CA mAP@50:95  CA mAP@50  \
Base State       Target State  Epochs                                      
Combined Dataset m0 Validation 1       0.024069      0.235842   0.708819   
                               2       0.114868      0.333752   0.797480   
                               3       0.122533      0.350679   0.800179   
                               4       0.168060      0.389262   0.824825   
                               5       0.165780      0.403930   0.858498   
...                                         ...           ...        ...   
                               71      0.342655      0.504884   0.875768   
                               72      0.308853      0.502321   0.876625   
                               73      0.321631      0.502116   0.875347   
                               74      0.316729      0.504168   0.875359   
                               75      0.318979      0.501160   0.875234   

                                       CA mAP@75  
Base State       Target State  Epochs             
Combined Dataset m0 Validation 1        0.064690  
                               2        0.216422  
                               3        0.206179  
                               4        0.310361  
                               5        0.324516  
...                                          ...  
                               71       0.538926  
                               72       0.535460  
                               73       0.548125  
                               74       0.543757  
                               75       0.545120  

[75 rows x 10 columns]

In [11]:
ca_mAP_df = saved_df.reset_index()[["Epochs", "CA mAP@50"]].set_index("Epochs")
display(ca_mAP_df)


,CA mAP@50
Epochs,
1,0.708819
2,0.797480
3,0.800179
4,0.824825
5,0.858498
...,...
71,0.875768
72,0.876625
73,0.875347


In [ ]:
# # Filter to include only epochs up to 50
# ca_mAP_df = ca_mAP_df[ca_mAP_df.index <= 50]
# Sort by CA mAP@50 values
sorted_df = ca_mAP_df.sort_values(by="CA mAP@50", ascending=False)
# display(sorted_df)

In [16]:
best_epoch = sorted_df.index[0]
print(f"Best epoch: {best_epoch}")

Best epoch: 21
